# 01 — UCI Heart Disease: Initial Data Analysis

**Industry context.** Clinical triage / cardiovascular risk decision support. We use the public UCI Heart Disease dataset (no PHI) to prototype a risk-stratification pipeline.

**Lineage.** The Initial Data Analysis (IDA) discipline followed here — distribution shape, missing/sentinel scan, group-level rates, chi-square + Cramer's V, explicit limitations — is adapted from Project 2 (Data & Statistical Reasoning, UCI Bank Marketing) and is the foundation for reproducible analytic work (Lusa et al., 2024).

**Educational artifact only. Not for clinical use.**

In [ ]:
import sys, os
from pathlib import Path
# Make the project root importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from src.data_loader import (
    load_heart_disease,
    FEATURE_DICTIONARY,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COL,
)

pd.set_option('display.max_columns', 50)
np.random.seed(42)

## 1. Load and inspect

In [ ]:
df = load_heart_disease()
print('Shape:', df.shape)
df.head()

In [ ]:
# Feature dictionary
for k, v in FEATURE_DICTIONARY.items():
    print(f'{k:10s}  {v}')

In [ ]:
df.describe(include='all').T

## 2. Missing / sentinel scan
Lineage: P2 — categorical sentinels (e.g. `unknown` in Bank Marketing) and NaNs were enumerated before any inference.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(missing if len(missing) else '(none)')

## 3. Target balance

In [ ]:
counts = df[TARGET_COL].value_counts().sort_index()
rates = (counts / counts.sum() * 100).round(2)
summary = pd.DataFrame({'count': counts, 'percent': rates})
summary.index = ['no disease (0)', 'disease (1)']
summary

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(summary.index, summary['count'], color=['#4C9AFF', '#FF6B6B'])
ax.set_title('Figure 1. Target class balance')
ax.set_ylabel('Patient count')
plt.tight_layout(); plt.show()

## 4. Numeric features — distributions and skew
Lineage: P2 reported skew for the variables most relevant to client-level inference; we do the same here for clinical variables.

In [ ]:
skew_table = df[NUMERIC_FEATURES].skew().round(3).rename('skew').to_frame()
skew_table

In [ ]:
NUMERIC_UNITS = {
    "age":      "years",
    "trestbps": "resting BP (mm Hg)",
    "chol":     "serum cholesterol (mg/dL)",
    "thalach":  "max heart rate (bpm)",
    "oldpeak":  "ST depression vs rest (mm)",
}

fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(3 * len(NUMERIC_FEATURES), 3.4))
for ax, col in zip(axes, NUMERIC_FEATURES):
    ax.hist(df[col].dropna(), bins=25, color='#4C9AFF', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(NUMERIC_UNITS.get(col, col))
    ax.set_ylabel('patients')
fig.suptitle('Figure 2. Numeric-feature distributions')
plt.tight_layout(); plt.show()

## 5. Disease rate by categorical feature
Lineage: P2 plotted subscription rate by job with the overall rate as a reference line. We apply the same idea per clinical category.

In [ ]:
CATEGORY_LABELS = {
    "sex":     {0: "female", 1: "male"},
    "cp":      {1: "typical angina", 2: "atypical angina", 3: "non-anginal", 4: "asymptomatic"},
    "fbs":     {0: "fasting BG <=120", 1: "fasting BG >120"},
    "restecg": {0: "normal", 1: "ST-T abnormality", 2: "LV hypertrophy"},
    "exang":   {0: "no exercise angina", 1: "exercise angina"},
    "slope":   {1: "upsloping ST", 2: "flat ST", 3: "downsloping ST"},
    "ca":      {0: "0 vessels", 1: "1 vessel", 2: "2 vessels", 3: "3 vessels"},
    "thal":    {3: "normal", 6: "fixed defect", 7: "reversible defect"},
}

overall = df[TARGET_COL].mean()
for col in CATEGORICAL_FEATURES:
    rates = df.groupby(col)[TARGET_COL].mean().sort_values()
    labels = [f"{CATEGORY_LABELS.get(col, {}).get(k, k)} ({k})" for k in rates.index]
    fig, ax = plt.subplots(figsize=(6, 0.4 * max(3, len(rates)) + 1))
    ax.barh(labels, rates.values * 100, color="#5BA37F")
    ax.axvline(overall * 100, color="red", linestyle="--", label=f"overall: {overall*100:.1f}%")
    ax.set_xlabel("Disease rate (%)")
    ax.set_ylabel(col)
    ax.set_title(f"Disease rate by {col}")
    ax.legend(loc="lower right")
    plt.tight_layout(); plt.show()

## 6. Chi-square test of independence (categorical features vs target)
Lineage: P2 used Pearson's chi-square + Cramer's V to test association between a categorical predictor and a binary outcome. We apply the same test family per categorical clinical feature, with Bonferroni-style awareness that we are running several tests.

- $H_0$: outcome is independent of the feature.
- $H_1$: outcome is associated with the feature.
- $\alpha = 0.05$. Cramer's V is the practical-magnitude effect size (≈0.10 small / 0.30 medium / 0.50 large; Cohen, 1988).

In [ ]:
def cramers_v(table: np.ndarray) -> float:
    chi2, _, _, _ = stats.chi2_contingency(table)
    n = table.sum()
    r, k = table.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1)))) if min(r, k) > 1 else float('nan')

rows = []
for col in CATEGORICAL_FEATURES:
    sub = df[[col, TARGET_COL]].dropna()
    table = pd.crosstab(sub[col], sub[TARGET_COL]).values
    if table.shape[0] < 2 or table.shape[1] < 2:
        continue
    chi2, p, dof, expected = stats.chi2_contingency(table)
    rows.append({
        'feature': col,
        'chi2': round(chi2, 3),
        'dof': dof,
        'p_value': p,
        'min_expected': round(expected.min(), 2),
        'cramers_v': round(cramers_v(table), 3),
    })
chi_df = pd.DataFrame(rows).sort_values('cramers_v', ascending=False).reset_index(drop=True)
chi_df

**Reading the table.** Features with `p_value < 0.05` and `min_expected ≥ 5` show statistically reliable association with disease outcome. Cramer's V indicates *how strong* the association is in practice. Expect `cp` (chest pain type), `thal`, `exang` (exercise-induced angina), and `ca` (number of vessels) to top this table — these are clinically well-established correlates of cardiovascular disease.

## 7. Correlation heatmap (numeric block)
Lineage: P2 used a correlation heatmap to flag collinearity in the macro-economic block. Same diagnostic here for numeric clinical features.

In [ ]:
corr = df[NUMERIC_FEATURES + [TARGET_COL]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.columns)), corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title('Figure 3. Pearson correlation of numeric features')
plt.tight_layout(); plt.show()

## 8. Limitations and Potential Bias
Lineage: P2 included an explicit Limitations & Bias section *before* presenting model results. The same discipline applies here, made concrete to the healthcare setting.

**Limitation — small, single-source cohort.** UCI Heart Disease aggregates ~300 records from a small set of clinical sites (Cleveland, Hungarian, Switzerland, VA Long Beach). Findings on this cohort will not generalise to populations with different demographic mixes, comorbidity profiles, or measurement standards. Any model trained here is for prototyping the pipeline, not for clinical inference.

**Bias — selection and recording.** Records reflect patients who *presented* at the contributing centres and consented to invasive testing (cardiac catheterisation underlies several features). Patients who never reached a cardiology service are absent. Disease rates measured here are therefore conditional on referral and will be biased upward relative to the general population.

**Bias — demographic skew.** The cohort skews male and older. Any per-slice evaluation in later phases must explicitly disaggregate by `sex` and age band, mirroring the per-class disaggregated evaluation discipline adopted from Project 4 (Fashion-MNIST CNN with dropout) — where 'same headline accuracy, different per-class behaviour' was the key insight.

**Caveat — feature provenance.** Several predictors (e.g. `ca`, `thal`, `oldpeak`) come from invasive or stress tests. A real triage system cannot assume these are available at intake; the eventual deployment story would split features into 'cheap at intake' vs 'available after workup'. We flag this here but do not enforce it in the prototype.

## References
- Agresti, A. (2007). *An Introduction to Categorical Data Analysis* (2nd ed.). Wiley.
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence Erlbaum.
- Lusa, L. et al. (2024). Initial data analysis for longitudinal studies. *PLOS ONE*, 19(5).
- UCI Machine Learning Repository — Heart Disease Data Set (ID 45).